### 为什么选择 Iceberg？它解决了什么问题？
首先我先说说相比于hive的优势，
第一点就是iceberg支持原子性，Hive 的 INSERT OVERWRITE 不是原子的，下游可能看到部分数据。
第二点就是视图变更，我说一下传统hive的痛点吧，以前Hive 修改列名、类型或顺序非常困难且危险，例如要新增一个字段记录用户来源，或者修改某个字段的类型以容纳更大数值，传统方式通常需要重写整个表，对大数据量时，这意味这漫长的停机等待，
虽然可以通过一些手段，比如说通过临时表解决，但计算资源的消耗也不可忽视，导致修改结构的成本太高，团队不敢轻易调整模型，导致表结构里面存有大量无用字段。但是iceberg支持完整的视图变更
然后第三点是Iceberg支持隐藏分区，我又来说一下传统Hive分区的一些痛点，hive分区需要在表中每一行数据重复存储，这在数据量大时会消耗巨大的存储空间，然后是查询逻辑复杂容易出错，我们去查的时候需要知道底层的分区时按天还是按小时，
一旦查询条件与底层分区格式不完全匹配，就会导致全表扫描，再然后是演进很困难，比如以前按天的分区不适用了，要改成按小时，就必须重写整个文件，iceberg通过将分区信息存储在元数据文件中，解决了这些问题
然后再说说一些特别的功能，比如说iceberg支持时间旅行和多引擎支持等hive不具备的功能
当然虽然iceberg引入了很多优势，但也是有劣势的，它引入了额外的复杂度，如果优化不当可能会比hive更慢，效率更低


### Iceberg的表格式由哪些核心文件组成？
1. metadata文件，包含表的schema、分区规范、快照等信息。
2. 清单文件，记录数据文件的列表以及每个数据文件的分区、行数、文件大小等统计信息。
3. 数据文件，实际存储数据的文件。

### Iceberg 如何保证 ACID 事务？
Iceberg是通过乐观并发控制的：Iceberg假设冲突很少发生。首先，它的写操作会生成新的元数据文件和清单文件，提交时，会尝试原子性地更新版本元数据文件，如果检测到当前版本已被其他事务修改，则重试或失败，同时每次提交都会生成一个新的快照，旧快照依然存在，保证了读操作不会因为写操作而报错。

### Iceberg 的 Schema Evolution（模式演进）有哪些优势？

